In [1]:
import sqlite3

In [2]:
conn = sqlite3.connect('ravenstack.db')


In [3]:
cursor = conn.cursor()

In [4]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS churn_events (
    churn_event_id TEXT PRIMARY KEY,
    account_id TEXT,
    churn_date TEXT,
    reason_code TEXT,
    refund_amount_usd REAL,
    preceding_upgrade_flag INTEGER,
    preceding_downgrade_flag INTEGER,
    is_reactivation INTEGER,
    feedback_text TEXT,
    FOREIGN KEY (account_id) REFERENCES accounts(account_id)
)
''')

In [5]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS support_tickets (
    ticket_id TEXT PRIMARY KEY,
    account_id TEXT,
    submitted_at TEXT,
    closed_at TEXT,
    resolution_time_hours REAL,
    priority TEXT,
    first_response_time_minutes INTEGER,
    satisfaction_score REAL,
    escalation_flag INTEGER,
    FOREIGN KEY (account_id) REFERENCES accounts(account_id)
)
''')

In [6]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS feature_usage (
    usage_id TEXT PRIMARY KEY,
    subscription_id TEXT,
    usage_date TEXT,
    feature_name TEXT,
    usage_count INTEGER,
    usage_duration_secs INTEGER,
    error_count INTEGER,
    is_beta_feature INTEGER,
    FOREIGN KEY (subscription_id) REFERENCES subscriptions(subscription_id)
)
''')

In [7]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS subscriptions (
    subscription_id TEXT PRIMARY KEY,
    account_id TEXT,
    start_date TEXT,
    end_date TEXT,
    plan_tier TEXT,
    seats INTEGER,
    mrr_amount INTEGER,
    arr_amount INTEGER,
    is_trial INTEGER,
    upgrade_flag INTEGER,
    downgrade_flag INTEGER,
    churn_flag INTEGER,
    billing_frequency TEXT,
    auto_renew_flag INTEGER,
    FOREIGN KEY (account_id) REFERENCES accounts(account_id)
)
''')

In [8]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS accounts (
    account_id TEXT PRIMARY KEY,
    account_name TEXT,
    industry TEXT,
    country TEXT,
    signup_date TEXT,
    referral_source TEXT,
    plan_tier TEXT,
    seats INTEGER,
    is_trial INTEGER,
    churn_flag INTEGER
)
''')

In [9]:
cursor.execute("DROP TABLE IF EXISTS hurn_events;")
conn.commit()

In [10]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('churn_events',), ('support_tickets',), ('feature_usage',), ('subscriptions',), ('accounts',)]


In [11]:
import pandas as pd

accounts = pd.read_csv("accounts.csv")
subscriptions = pd.read_csv("subscriptions.csv")
feature_usage = pd.read_csv("feature_usage.csv")
support_tickets = pd.read_csv("support_tickets.csv")
churn_events = pd.read_csv("churn_events.csv")

In [12]:
cursor.execute("DELETE FROM churn_events")
cursor.execute("DELETE FROM support_tickets")
cursor.execute("DELETE FROM feature_usage")
cursor.execute("DELETE FROM subscriptions")
cursor.execute("DELETE FROM accounts")

conn.commit()

In [13]:
feature_usage = feature_usage.drop_duplicates(subset=["usage_id"], keep="first")

In [14]:
print(feature_usage["usage_id"].duplicated().sum())

0


In [15]:
accounts.to_sql("accounts", conn, if_exists="append", index=False)
subscriptions.to_sql("subscriptions", conn, if_exists="append", index=False)


5000

In [16]:
cursor.execute("SELECT COUNT(*) FROM subscriptions;")
print(cursor.fetchall())

[(5000,)]


In [17]:
feature_usage.to_sql("feature_usage", conn, if_exists="append", index=False)

support_tickets.to_sql("support_tickets", conn, if_exists="append", index=False)

churn_events.to_sql("churn_events", conn, if_exists="append", index=False)

600

In [18]:
cursor.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT account_id) AS unique_accounts
FROM accounts
""")

print(cursor.fetchall())

[(500, 500)]


In [19]:
query = """
SELECT
    COUNT(*) AS total_accounts,
    SUM(churn_flag) AS churned_accounts,
    ROUND(SUM(churn_flag) * 100.0 / COUNT(*), 2) AS churn_rate
FROM accounts;
"""

df = pd.read_sql_query(query, conn)
print(df)

   total_accounts  churned_accounts  churn_rate
0             500               110        22.0


In [20]:
query = ' ' ' SELECT industry from accounts; ' ' '
df= pd.read_sql_query(query,conn)
df

,industry
0,EdTech
1,FinTech
2,DevTools
3,HealthTech
4,HealthTech
...,...
495,DevTools
496,FinTech
497,DevTools
498,EdTech


In [21]:
query = ''' SELECT
    industry,
    COUNT(*) AS total_accounts,
    SUM(churn_flag) AS churned_accounts,
    ROUND( SUM(churn_flag)* 100.0 / COUNT(*), 2) AS churn_rate
FROM accounts
GROUP BY industry
ORDER BY churn_rate DESC;'''
pd.read_sql_query(query,conn)

,industry,total_accounts,churned_accounts,churn_rate
0,DevTools,113,35,30.97
1,FinTech,112,25,22.32
2,HealthTech,96,21,21.88
3,EdTech,79,13,16.46
4,Cybersecurity,100,16,16.00


In [22]:
query = '''SELECT
    churn_flag,
    AVG(ticket_count) AS avg_ticket_count
FROM (
    SELECT
        a.account_id,
        a.churn_flag,
        COUNT(s.ticket_id) AS ticket_count
    FROM accounts a
    LEFT JOIN support_tickets s
        ON a.account_id = s.account_id
    GROUP BY a.account_id, a.churn_flag
)
GROUP BY churn_flag;'''
df=pd.read_sql_query(query,conn)
df

,churn_flag,avg_ticket_count
0,0,4.020513
1,1,3.927273


query 1

In [23]:
query = '''SELECT *
FROM subscriptions
LIMIT 5;'''
df=pd.read_sql_query(query,conn)
df

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,0,0,0,1,monthly,1
1,S-0f6f44,A-9b9fe9,2024-06-11,None,Pro,17,833,9996,0,0,0,0,monthly,1
2,S-51c0d1,A-659280,2024-11-25,None,Enterprise,62,0,0,1,1,0,0,annual,0
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,0,0,0,1,monthly,1
4,S-cff5a2,A-ba6516,2024-01-10,None,Enterprise,27,5373,64476,0,0,0,0,monthly,1


query 2

In [24]:
query = """
WITH usage_summary AS (
    SELECT
        subscription_id,
        SUM(usage_count) AS total_usage
    FROM feature_usage
    GROUP BY subscription_id
)

SELECT
    s.churn_flag,
    AVG(u.total_usage) AS avg_feature_usage
FROM usage_summary u
JOIN subscriptions s
    ON u.subscription_id = s.subscription_id
GROUP BY s.churn_flag;
"""

df = pd.read_sql_query(query, conn)
print(df)

   churn_flag  avg_feature_usage
0           0          50.521186
1           1          49.244306


Query 3


In [51]:
query = """

WITH ranked_churn AS (
    SELECT
        account_id,
        plan_tier,
        mrr_amount,
        churn_flag,
        RANK() OVER (
            PARTITION BY plan_tier 
            ORDER BY mrr_amount DESC
        ) AS revenue_rank
    FROM subscriptions
    WHERE churn_flag = 1
)
SELECT *
FROM ranked_churn
WHERE revenue_rank <= 3;
"""

df = pd.read_sql_query(query, conn)
print(df)

  account_id   plan_tier  mrr_amount  churn_flag  revenue_rank
0   A-18793f       Basic        2812           1             1
1   A-0cc442       Basic        1843           1             2
2   A-462d45       Basic        1691           1             3
3   A-118f1c  Enterprise       25472           1             1
4   A-d4e0d4  Enterprise       23283           1             2
5   A-ce550d  Enterprise       21691           1             3
6   A-104f1a         Pro        6027           1             1
7   A-5a215a         Pro        4263           1             2
8   A-5a215a         Pro        4263           1             2


In [53]:
query ='''SELECT subscription_id, account_id, plan_tier, mrr_amount, 
       start_date, end_date, churn_flag
FROM subscriptions
WHERE account_id = 'A-5a215a';'''

df = pd.read_sql_query(query, conn)
print(df)

   subscription_id account_id   plan_tier  mrr_amount  start_date    end_date  \
0         S-8324d4   A-5a215a         Pro           0  2024-07-13  2024-08-03   
1         S-838330   A-5a215a         Pro           0  2024-06-20        None   
2         S-93f835   A-5a215a         Pro        4263  2024-02-11        None   
3         S-878fe0   A-5a215a         Pro        4263  2024-12-10        None   
4         S-5da6e7   A-5a215a         Pro        4263  2024-07-29  2024-09-19   
5         S-5bb58e   A-5a215a       Basic        1653  2024-01-31        None   
6         S-2223bf   A-5a215a         Pro        4263  2024-10-23  2024-11-26   
7         S-3781d0   A-5a215a  Enterprise       17313  2024-05-10  2024-12-06   
8         S-75cba6   A-5a215a  Enterprise       17313  2024-09-18        None   
9         S-704cf3   A-5a215a       Basic        1653  2024-11-24        None   
10        S-d8fa45   A-5a215a       Basic        1653  2024-10-18        None   
11        S-ca1cb9   A-5a215

Query 4

In [26]:
query = """
SELECT account_id, plan_tier, mrr_amount, start_date, end_date, churn_flag
FROM subscriptions
WHERE account_id = 'A-5a215a'
ORDER BY start_date DESC;
"""

df = pd.read_sql_query(query, conn)
print(df)

   account_id   plan_tier  mrr_amount  start_date    end_date  churn_flag
0    A-5a215a         Pro        4263  2024-12-10        None           0
1    A-5a215a       Basic        1653  2024-11-24        None           0
2    A-5a215a         Pro        4263  2024-10-23  2024-11-26           1
3    A-5a215a       Basic        1653  2024-10-18        None           0
4    A-5a215a  Enterprise       17313  2024-09-18        None           0
5    A-5a215a  Enterprise       17313  2024-09-18        None           0
6    A-5a215a  Enterprise       17313  2024-09-16  2024-11-13           1
7    A-5a215a         Pro        4263  2024-07-29  2024-09-19           1
8    A-5a215a         Pro           0  2024-07-13  2024-08-03           1
9    A-5a215a         Pro        4263  2024-07-10        None           0
10   A-5a215a         Pro           0  2024-06-20        None           0
11   A-5a215a  Enterprise       17313  2024-05-27  2024-12-30           1
12   A-5a215a  Enterprise       17313 

In [27]:
conn.commit()

In [28]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("ravenstack.db")

tables = ["accounts", "subscriptions", "feature_usage", "support_tickets", "churn_events"]

for t in tables:
    df = pd.read_sql_query(f"SELECT * FROM {t}", conn)
    df.to_csv(f"{t}.csv", index=False)

conn.close()

In [29]:
import pandas as pd

In [30]:
import sqlite3

In [31]:
conn = sqlite3.connect('ravenstack.db')
cursor=conn.cursor()

query 5

In [38]:
query = """
WITH latest_sub AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY account_id 
            ORDER BY start_date DESC
        ) AS rn
    FROM subscriptions s
),

current_sub AS (
    SELECT *
    FROM latest_sub
    WHERE rn = 1
),

ranked_churn AS (
    SELECT
        a.account_id,
        a.account_name,
        cs.plan_tier,
        a.churn_flag,
        cs.mrr_amount AS revenue,
        RANK() OVER (
            PARTITION BY cs.plan_tier 
            ORDER BY cs.mrr_amount DESC
        ) AS revenue_rank
    FROM accounts a
    JOIN current_sub cs ON a.account_id = cs.account_id
    WHERE a.churn_flag = 1
)

SELECT *
FROM ranked_churn
ORDER BY plan_tier, revenue_rank;
"""

df = pd.read_sql_query(query, conn)
print(df)

    account_id account_name plan_tier  churn_flag  revenue  revenue_rank
0     A-30b4ca   Company_23     Basic           1     1634             1
1     A-902ef7  Company_270     Basic           1     1577             2
2     A-e83938  Company_111     Basic           1     1026             3
3     A-b2225d   Company_50     Basic           1      893             4
4     A-7f6b86  Company_299     Basic           1      855             5
..         ...          ...       ...         ...      ...           ...
105   A-ffdfd5  Company_365       Pro           1      147            30
106   A-69fad4  Company_103       Pro           1        0            31
107   A-7dacce    Company_8       Pro           1        0            31
108   A-95b24a  Company_117       Pro           1        0            31
109   A-ab438f  Company_233       Pro           1        0            31

[110 rows x 6 columns]


In [36]:
for table in ['accounts', 'subscriptions', 'feature_usage', 'support_tickets', 'churn_events']:
    print(f"\n--- {table} ---")
    print(pd.read_sql_query(f"PRAGMA table_info({table});", conn))


--- accounts ---
   cid             name     type  notnull dflt_value  pk
0    0       account_id     TEXT        0       None   1
1    1     account_name     TEXT        0       None   0
2    2         industry     TEXT        0       None   0
3    3          country     TEXT        0       None   0
4    4      signup_date     TEXT        0       None   0
5    5  referral_source     TEXT        0       None   0
6    6        plan_tier     TEXT        0       None   0
7    7            seats  INTEGER        0       None   0
8    8         is_trial  INTEGER        0       None   0
9    9       churn_flag  INTEGER        0       None   0

--- subscriptions ---
    cid               name     type  notnull dflt_value  pk
0     0    subscription_id     TEXT        0       None   1
1     1         account_id     TEXT        0       None   0
2     2         start_date     TEXT        0       None   0
3     3           end_date     TEXT        0       None   0
4     4          plan_tier     T

Query 5

In [48]:
import pandas as pd

# Load tables from SQLite
accounts = pd.read_sql_query("SELECT * FROM accounts", conn)
subscriptions = pd.read_sql_query("SELECT * FROM subscriptions", conn)

# -----------------------------
# Step 1: latest_sub (ROW_NUMBER)
# -----------------------------

# Convert to datetime
subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])

# Sort by account_id and latest start_date
subscriptions = subscriptions.sort_values(
    ["account_id", "start_date"],
    ascending=[True, False]
)

# Create row number
subscriptions["rn"] = (
    subscriptions.groupby("account_id")
    .cumcount() + 1
)

# -----------------------------
# Step 2: current_sub
# -----------------------------
current_sub = subscriptions[subscriptions["rn"] == 1].copy()

# -----------------------------
# Step 3: Join with accounts
# -----------------------------
ranked_churn = accounts.merge(
    current_sub,
    on="account_id",
    how="inner"
)

# -----------------------------
# Step 4: Keep only churned accounts
# -----------------------------
ranked_churn = ranked_churn[
    ranked_churn["churn_flag_x"] == 1
].copy()

# -----------------------------
# Step 5: Rename revenue column
# -----------------------------
ranked_churn["revenue"] = ranked_churn["mrr_amount"]

# -----------------------------
# Step 6: Create SQL RANK()
# -----------------------------
ranked_churn["revenue_rank"] = (
    ranked_churn.groupby("plan_tier_y")["revenue"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# -----------------------------
# Step 7: Select required columns
# -----------------------------
# result = ranked_churn[
#     [
#         "account_id",
#         "account_name",
#         "plan_tier",
#         "churn_flag",
#         "revenue",
#         "revenue_rank"
#     ]
# ].sort_values(
#     ["plan_tier", "revenue_rank"]
# )

# print(result)
result = ranked_churn[
    [
        "account_id",
        "account_name",
        "plan_tier_y",
        "churn_flag_x",
        "revenue",
        "revenue_rank"
    ]
].rename(columns={
    "plan_tier_y": "plan_tier",
    "churn_flag_x": "churn_flag"
}).sort_values(
    ["plan_tier", "revenue_rank"]
)

print(result)

    account_id account_name plan_tier  churn_flag  revenue  revenue_rank
23    A-30b4ca   Company_23     Basic           1     1634             1
270   A-902ef7  Company_270     Basic           1     1577             2
111   A-e83938  Company_111     Basic           1     1026             3
50    A-b2225d   Company_50     Basic           1      893             4
299   A-7f6b86  Company_299     Basic           1      855             5
..         ...          ...       ...         ...      ...           ...
365   A-ffdfd5  Company_365       Pro           1      147            30
8     A-7dacce    Company_8       Pro           1        0            31
103   A-69fad4  Company_103       Pro           1        0            31
117   A-95b24a  Company_117       Pro           1        0            31
233   A-ab438f  Company_233       Pro           1        0            31

[110 rows x 6 columns]


In [42]:
print(ranked_churn.columns.tolist())

['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier_x', 'seats_x', 'is_trial_x', 'churn_flag_x', 'subscription_id', 'start_date', 'end_date', 'plan_tier_y', 'seats_y', 'mrr_amount', 'arr_amount', 'is_trial_y', 'upgrade_flag', 'downgrade_flag', 'churn_flag_y', 'billing_frequency', 'auto_renew_flag', 'rn', 'revenue']


In [46]:
print(accounts.columns.tolist())
print(current_sub.columns.tolist())
print(ranked_churn.columns.tolist())

['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']
['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag', 'rn']
['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier_x', 'seats_x', 'is_trial_x', 'churn_flag_x', 'subscription_id', 'start_date', 'end_date', 'plan_tier_y', 'seats_y', 'mrr_amount', 'arr_amount', 'is_trial_y', 'upgrade_flag', 'downgrade_flag', 'churn_flag_y', 'billing_frequency', 'auto_renew_flag', 'rn', 'revenue', 'revenue_rank']
